# Climate Change Trend Analysis and Forecasting
## Week 6: Notebook Finalization and Project Synthesis

### Project Objective
The goal of this end-to-end analytical project is to ingest open Greenhouse Gas (GHG) emissions datasets, engineer robust time-series features, evaluate supervised classical machine learning models, and implement structural time-series forecasting ($ETS(A,A_d,N)$ Holt Damped Trend) to project emissions 20 years into the future. 

### Countries of Focus
The analysis is restricted strictly to 10 representative countries: 
**China, USA, India, Russia, Japan, Germany, Brazil, United Kingdom, South Africa, and Australia.**

### Chronological Roadmap
1. **Week 1-2**: Dataset Profiling & Feature Engineering (`ghg_features.csv`).
2. **Week 3**: Evaluation of Baseline Supervised Regression Frameworks (Naive, Linear Regression, Random Forest).
3. **Week 4**: Implementation of $ETS(A,A_d,N)$ Holt Damped Trend framework to generate out-of-sample forecasts to 2043.
4. **Week 5**: Implementation of policy intervention strategies (Business-as-Usual, Moderate Mitigation, Aggressive Mitigation) stored in `scenario_projections_2.csv`.
5. **Week 6**: Complete structural clean-up, execution verification, and Streamlit application setup.

### Environment and Libraries
We load the core analytics toolkit required for processing dataframes (`pandas`, `numpy`), rendering metrics (`scikit-learn`), configuring time-series modeling engines (`statsmodels`), and displaying dynamic graphics (`matplotlib`, `seaborn`).

In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 14

print("Environment setup completed successfully.")

### Historical Dataset Overview
We query the finalized configuration rows derived from `owid-co2-data.csv` to ensure structural alignment with our historical tracking windows starting from 1990.

In [ ]:
# Load verbatim file tracking historical metrics using the correct filename
df_owid = pd.read_csv("owid-co2-data.csv")

# Confirm dataset metrics and core structural presence
print(f"Shape of historical dataset: {df_owid.shape}")
print(df_owid[["country", "year", "co2"]].head(5))

### Supervised Regression Task Framework
* **Task Goal**: Given a matrix of feature characteristics $X$ up to year $Y$ for country $C$, project the specific single target point $Y+1$ for $\text{CO}_2$ emissions.
* **Target Feature**: `co2` (Million tonnes)
* **Input Features**: `co2_lag1`, `co2_lag2`, `co2_lag3`, `co2_5yr_rolling_mean`, `years_since_1990`, `co2_yoy_pct_change`
* **Splitting Mechanism**: **Temporal Splitting** (Train: 1990–2018; Test: 2019–2023). Standard random splits are completely rejected here to safeguard against structural data leakage and explicitly analyze performance during the complex COVID-19 pandemic drop (2020).

In [ ]:
# Focus target list (mapped to exact names in the dataset)
target_countries = [
    "China",
    "United States",
    "India",
    "Russia",
    "Japan",
    "Germany",
    "Brazil",
    "United Kingdom",
    "South Africa",
    "Australia",
]

# Quick calculation functions for features over focus countries
ml_data = []
for country in target_countries:
    c_df = df_owid[df_owid["country"] == country].sort_values("year").copy()
    c_df = c_df[c_df["year"] >= 1990]

    # Engineering baseline model rows safely
    c_df["years_since_1990"] = c_df["year"] - 1990
    c_df["co2_5yr_rolling_mean"] = c_df["co2"].rolling(window=5).mean()
    c_df["co2_lag1"] = c_df["co2"].shift(1)
    c_df["co2_lag2"] = c_df["co2"].shift(2)
    c_df["co2_lag3"] = c_df["co2"].shift(3)
    c_df["co2_yoy_pct_change"] = c_df["co2"].pct_change()

    ml_data.append(c_df)

df_ml = pd.concat(ml_data, ignore_index=True).dropna(
    subset=["co2", "co2_lag1", "co2_lag2", "co2_lag3", "co2_5yr_rolling_mean"]
)

# Container for statistical results storage
metrics_summary = []

for country in target_countries:
    country_data = df_ml[df_ml["country"] == country]

    train_set = country_data[country_data["year"] <= 2018]
    test_set = country_data[
        (country_data["year"] >= 2019) & (country_data["year"] <= 2023)
    ]

    features = [
        "years_since_1990",
        "co2_5yr_rolling_mean",
        "co2_lag1",
        "co2_lag2",
        "co2_lag3",
    ]

    X_train, y_train = train_set[features], train_set["co2"]
    X_test, y_test = test_set[features], test_set["co2"]

    # 1. Naive Baseline (Next year = current year)
    y_pred_naive = X_test["co2_lag1"]
    mae_naive = mean_absolute_error(y_test, y_pred_naive)
    rmse_naive = np.sqrt(mean_squared_error(y_test, y_pred_naive))

    # 2. Linear Regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    mae_lr = mean_absolute_error(y_test, y_pred_lr)
    rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

    # 3. Random Forest Regressor
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    mae_rf = mean_absolute_error(y_test, y_pred_rf)
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

    metrics_summary.append(
        {
            "Country": country,
            "Baseline MAE": mae_naive,
            "Baseline RMSE": rmse_naive,
            "LR MAE": mae_lr,
            "LR RMSE": rmse_lr,
            "RF MAE": mae_rf,
            "RF RMSE": rmse_rf,
        }
    )

df_metrics = pd.DataFrame(metrics_summary)
print("Supervised Learning Baselines Evaluated Successfully.")
df_metrics.head(5)

### $ETS(A,A_d,N)$ Structural Framework
The Holt Damped Trend model isolates non-seasonal trajectories with parameter protections:
* **Error ($A$)**: Additive residuals.
* **Trend ($A_d$)**: Additive damped trend. The trend component decays toward zero via a dampening coefficient $\phi$ ($0 < \phi < 1$).
* **Seasonality ($N$)**: None (annual aggregation scales exhibit zero sub-yearly cycles).

#### Rationale for Emissions Forecasting
Unlike unconstrained linear architectures, the tracking index bounded by $\phi$ structurally restricts unbounded emissions expansion. This matches macroeconomic transformations, efficiency benchmarks, and legal policy shifts across advanced industrial regimes (e.g., UK, Germany).

In [ ]:
ets_holdout_results = []
forecast_2043_storage = {}

for country in target_countries:
    country_data = df_ml[df_ml["country"] == country].sort_values("year")

    # Split for holdout scoring calculation
    train_series = country_data[country_data["year"] <= 2018]["co2"].values
    holdout_actuals = country_data[
        (country_data["year"] >= 2019) & (country_data["year"] <= 2023)
    ]["co2"].values

    # Fit the ETS model
    ets_model = ExponentialSmoothing(
        train_series, trend="add", damped_trend=True, seasonal=None
    )
    ets_fit = ets_model.fit(optimized=True)

    # Forecast across the test timeline
    ets_test_pred = ets_fit.forecast(len(holdout_actuals))
    mae_ets = mean_absolute_error(holdout_actuals, ets_test_pred)
    rmse_ets = np.sqrt(mean_squared_error(holdout_actuals, ets_test_pred))

    ets_holdout_results.append(
        {"Country": country, "ETS MAE": mae_ets, "ETS RMSE": rmse_ets}
    )

    # Complete refit on whole timeline up to 2024 for clean future forecasts
    full_series = country_data["co2"].values
    final_ets = ExponentialSmoothing(
        full_series, trend="add", damped_trend=True, seasonal=None
    ).fit(optimized=True)

    # Forecast 20 years forward (2025 to 2044)
    future_fc = final_ets.forecast(20)
    forecast_2043_storage[country] = future_fc

df_ets_metrics = pd.DataFrame(ets_holdout_results)
print("ETS Forecasting Architecture Fitted.")

### Structural What-If Scenario Formulations
* **Scenario A - Business-as-Usual (BAU)**: Unaltered projection utilizing the generated $ETS(A,A_d,N)$ model pathway.
* **Scenario B - Moderate Mitigation**: Applies an annual compounding drop of 2% relative to the BAU baseline starting from 2025.
* **Scenario C - Aggressive Mitigation**: Applies an annual compounding drop of 5% relative to the BAU baseline starting from 2025.

In [ ]:
# Adjusted to the exact filename in the working directory
df_scenarios = pd.read_csv("scenario_projections.csv")

print("Validating structural shape of projected parameters:")
print(df_scenarios.shape)

### Consolidated Error Metrics Table
We build a four-model validation map evaluated directly against the 2019–2023 structural shock window.

In [ ]:
# Combine standard evaluation metrics
df_performance = pd.merge(df_metrics, df_ets_metrics, on="Country")

# Establish explicit ordering configuration
clean_cols = [
    "Country",
    "Baseline MAE",
    "LR MAE",
    "RF MAE",
    "ETS MAE",
    "Baseline RMSE",
    "LR RMSE",
    "RF RMSE",
    "ETS RMSE",
]
df_performance = df_performance[clean_cols]


# Determine optimization winner per row
def find_best_model(row):
    maes = {
        "Baseline": row["Baseline MAE"],
        "Linear Regression": row["LR MAE"],
        "Random Forest": row["RF MAE"],
        "ETS": row["ETS MAE"],
    }
    return min(maes, key=maes.get)


df_performance["Best Model (MAE)"] = df_performance.apply(
    find_best_model, axis=1
)

# Display completed verification frame
df_performance

### Strategic Analysis Summary
1. **Model Resiliency**: Supervised machine learning algorithms (Linear Regression and Random Forest) show signs of overfitting on previous emission baselines. They struggle to accurately capture the sudden, real-world structural break caused by the 2020 COVID-19 pandemic drop.
2. **ETS Adaptability**: The $ETS(A,A_d,N)$ framework provides smoother, trend-damped projections. This reduces error propagation over longer forecast horizons compared to unconstrained polynomial regression trends.

### Generating the Production Dashboard Script
We use the `%%writefile` magic command to save our app code into a script called `app.py`.

In [1]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="GHG Forecasting Dashboard", layout="wide")

st.title("🌍 Climate Change Trend Analysis & Scenario Forecasting Dashboard")
st.markdown("---")

# Load compiled assets safely from your local directory
@st.cache_data
def load_internal_data():
    hist = pd.read_csv("owid-co2-data.csv")
    scen = pd.read_csv("scenario_projections.csv")
    return hist, scen

df_hist, df_scen = load_internal_data()

# Metric sidebar controls
st.sidebar.header("Navigation & Filters")
target_countries = sorted(list(df_scen['country'].unique()))
selected_country = st.sidebar.selectbox("Select Target Country Focus", target_countries)

# Section 1: Key Performance Indicators
st.subheader(f"📊 Macro Metrics Profile: {selected_country}")
c_hist = df_hist[df_hist['country'] == selected_country].sort_values('year')
latest_year = int(c_hist['year'].max())
latest_co2 = c_hist[c_hist['year'] == latest_year]['co2'].values[0]

kpi1, kpi2 = st.columns(2)
kpi1.metric(label=f"Historical CO2 Emissions ({latest_year})", value=f"{latest_co2:,.2f} MtCO₂")
kpi2.metric(label="Forecast Window Horizon", value="2025 — 2044")

# Section 2: Projections Plotting Frame
st.subheader("🔮 What-If Mitigation Pathways (2025–2044)")
c_scen = df_scen[df_scen['country'] == selected_country]

fig = px.line(c_scen, x='year', y='co2_projected', color='scenario',
              labels={'co2_projected': 'CO₂ Emissions (Mt)', 'year': 'Timeline Year'},
              title=f"Forecast Scenarios Comparison Vector for {selected_country}")
st.plotly_chart(fig, use_container_width=True)

st.markdown("🔒 *Dashboard successfully configured and running locally via VS Code terminal.*")

Writing app.py
